# MonopolyZero — ASU-guided training

Restores an optional run bundle, imports ASU expert shards, bootstraps from PPO-v2, trains on the selected accelerator, and always exports a resumable result archive. Set `generations` to `0` to export the ASU-trained bootstrap checkpoint before starting expensive self-play.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, tarfile, urllib.request

CONTENT = Path(os.environ.get('MONOPOLYZERO_CONTENT', '/content'))
JOB = {
    'commit': '52b75d592df6e448d6de5afbd2ca5a360c61bc16',
    'generations': 1,
    'device': 'auto',
}
job_path = CONTENT / 'monopolyzero-train-job.json'
if job_path.exists():
    JOB.update(json.loads(job_path.read_text()))
assert re.fullmatch(r'[0-9a-f]{40}', JOB['commit'])
assert int(JOB['generations']) >= 0
RUN_DIR = CONTENT / 'monopolyzero-ppo-plus-v2-v1'
RESULT = CONTENT / 'monopolyzero-result.tar.gz'
STATUS_PATH = CONTENT / 'monopolyzero-train-status.json'
print(json.dumps(JOB, indent=2, sort_keys=True))

In [ ]:
import torch
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
STATUS = {
    **JOB, 'state': 'running', 'cpu_count': os.cpu_count(),
    'ram_gib': round(ram_gib, 2), 'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
STATUS_PATH.write_text(json.dumps(STATUS, indent=2, sort_keys=True) + '\n')
print(json.dumps(STATUS, indent=2, sort_keys=True))

In [ ]:
repository_archive = CONTENT / 'DeepRL_Monopoly.tar.gz'
if repository_archive.exists():
    with tarfile.open(repository_archive, 'r:gz') as archive:
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / 'DeepRL_Monopoly'
else:
    repository_archive = CONTENT / f"DeepRL_Monopoly-{JOB['commit']}.tar.gz"
    urllib.request.urlretrieve(
        f"https://codeload.github.com/Darkosxl/DeepRL_Monopoly/tar.gz/{JOB['commit']}",
        repository_archive,
    )
    with tarfile.open(repository_archive, 'r:gz') as archive:
        roots = {Path(member.name).parts[0] for member in archive.getmembers() if member.name}
        assert len(roots) == 1
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / roots.pop()
assert (REPOSITORY_ROOT / 'monopoly_bench').is_dir()
baseline_bundle = CONTENT / 'monopolyzero-baselines.tar.gz'
if not baseline_bundle.exists():
    raise FileNotFoundError('Upload /content/monopolyzero-baselines.tar.gz')
with tarfile.open(baseline_bundle, 'r:gz') as archive:
    archive.extractall(REPOSITORY_ROOT, filter='data')
resume_bundle = CONTENT / 'monopolyzero-resume.tar.gz'
if resume_bundle.exists():
    with tarfile.open(resume_bundle, 'r:gz') as archive:
        archive.extractall(CONTENT, filter='data')
    restored = CONTENT / 'run'
    if restored.exists():
        shutil.copytree(restored, RUN_DIR, dirs_exist_ok=True)
EXPERT_SHARDS = sorted({
    *CONTENT.glob('monopolyzero-asu-*.npz'),
    *(CONTENT / 'asu').glob('*.npz'),
})
print(f'Repository: {REPOSITORY_ROOT} | ASU shards: {len(EXPERT_SHARDS)}')

In [ ]:
command = [
    sys.executable, '-m', 'monopoly_bench', 'train',
    '--run-dir', str(RUN_DIR),
    '--bootstrap-ppo', str(REPOSITORY_ROOT / 'artifacts/ppo_plus/ppo_hybrid_2000_v2.pt'),
    '--generations', str(JOB['generations']),
    '--device', str(JOB['device']),
]
if not (RUN_DIR / 'asu_expert.npz').exists():
    if not EXPERT_SHARDS:
        raise FileNotFoundError('Upload at least one ASU expert shard')
    command.extend(['--asu-expert-data', *map(str, EXPERT_SHARDS)])
try:
    subprocess.run(command, cwd=REPOSITORY_ROOT, check=True)
except Exception as exc:
    STATUS.update(state='failed', error=f'{type(exc).__name__}: {exc}')
    raise
else:
    STATUS.update(state='complete')
finally:
    temporary_result = RESULT.with_suffix('.tmp')
    with tarfile.open(temporary_result, 'w:gz') as archive:
        if RUN_DIR.exists():
            archive.add(RUN_DIR, arcname='run')
    temporary_result.replace(RESULT)
    STATUS['result_sha256'] = hashlib.sha256(RESULT.read_bytes()).hexdigest()
    temporary_status = STATUS_PATH.with_suffix('.tmp')
    temporary_status.write_text(json.dumps(STATUS, indent=2, sort_keys=True) + '\n')
    temporary_status.replace(STATUS_PATH)

In [ ]:
sys.path.insert(0, str(REPOSITORY_ROOT))
from monopoly_bench.model import MonopolyZeroNet
candidates = sorted((RUN_DIR / 'candidates').glob('generation_*.pt'))
checkpoints = sorted((RUN_DIR / 'checkpoints').glob('generation_*.pt'))
if candidates:
    model = MonopolyZeroNet.load_inference(candidates[-1])
    assert model.policy_head.out_features == 2958
print({
    'latest_candidate': str(candidates[-1]) if candidates else None,
    'latest_checkpoint': str(checkpoints[-1]) if checkpoints else None,
    'result': str(RESULT),
    'status': STATUS,
})